# BitCal-TTS: Bit-Calibrated Test-Time Scaling

**Full minimal experiment on Google Colab (T4 GPU)**

Runs 3 methods (fixed / adaptive / bitcal_tts) on GSM8K with
Qwen2.5-3B-Instruct at 4-bit quantization.

**Runtime estimate:** ~30-45 min for 50 items × 3 budgets × 3 methods on T4.

**Steps:**
1. Check GPU
2. Install dependencies
3. Clone repo
4. Run experiment
5. Analyze & plot results
6. Download results

In [ ]:
# Cell 1 — Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q -U transformers>=4.38.0 accelerate>=0.27.0 datasets>=2.18.0 matplotlib>=3.7.0 pyyaml scipy
print('Dependencies installed.')

In [ ]:
# Cell 3 — Clone repo
import os
if not os.path.exists('bitcal-tts'):
    !git clone https://github.com/Saibabu7770/bitcal-tts.git
else:
    !cd bitcal-tts && git pull origin main
%cd bitcal-tts
!pip install -e . -q
print('Repo ready.')

In [ ]:
# Cell 4 — Verify setup
!python -m bitcal_tts doctor

In [ ]:
# Cell 5 — Quick smoke test (1 item, 64 tokens) before full run
!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-3B-Instruct \
    --n-samples 1 \
    --budget 64 \
    --methods fixed \
    --source hf \
    --step-size 8
print('Smoke test done.')

In [ ]:
# Cell 6 — FULL EXPERIMENT
# 50 items x 3 budgets x 3 methods = 450 runs
# Estimated time: 30-45 min on T4 with 4-bit
#
# --min-tokens-before-halt 128  <- hard floor: model must generate
#   at least 128 tokens before the halt policy is allowed to fire.
#   This prevents premature stopping on math reasoning tasks.
import subprocess, sys, os
repo = next(r for r,d,_ in os.walk('/content') if 'scripts' in d and 'configs' in d)
os.chdir(repo)
print(f'Running from: {repo}')
!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-3B-Instruct \
    --n-samples 50 \
    --budget 256 512 1024 \
    --methods fixed adaptive bitcal_tts \
    --source hf \
    --step-size 16 \
    --min-tokens-before-halt 128 \
    --output-dir results/raw
print('Full experiment complete.')

In [ ]:
# Cell 7 — Analyze results and generate plots
!python scripts/analyze_results.py \
    --results-dir results/raw \
    --out-dir results/processed

In [ ]:
# Cell 8 — Display plots inline
from IPython.display import Image, display
import os

for fname in ['pareto_quality_tokens.png', 'accuracy_by_budget.png']:
    path = f'results/processed/{fname}'
    if os.path.exists(path):
        print(f'\n--- {fname} ---')
        display(Image(path))

In [ ]:
# Cell 9 — Print full summary table
import pandas as pd
df = pd.read_csv('results/processed/summary.csv')
print(df.to_string(index=False))

In [ ]:
# Cell 10 — Download all results as a zip
import shutil
from google.colab import files

shutil.make_archive('bitcal_tts_results', 'zip', '.', 'results')
files.download('bitcal_tts_results.zip')
print('Downloaded: bitcal_tts_results.zip')

## Optional: Push results back to GitHub

To save results directly to your GitHub repo, run the cell below.
You need a GitHub Personal Access Token with repo write access.

Create one at: https://github.com/settings/tokens
(Scopes needed: `Contents: Read and write`)

In [ ]:
# Cell 11 (optional) — Push results to GitHub
# Fill in your token below, then run this cell
GITHUB_TOKEN = ''  # paste your PAT here
GITHUB_EMAIL = 'Saibabu7770@users.noreply.github.com'

if GITHUB_TOKEN:
    !git config user.name 'Sai'
    !git config user.email '{GITHUB_EMAIL}'
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/Saibabu7770/bitcal-tts.git
    !git add results/
    !git commit -m 'Results: Colab T4 full experiment - Qwen2.5-3B 4bit GSM8K 50 items'
    !git push origin main
    print('Results pushed to GitHub!')
else:
    print('Skipped: set GITHUB_TOKEN to push results.')